# So sánh Feature Selection: Boruta vs Lasso — Communities and Crime

Tương tự `test_housing_lassovsboruta.ipynb` nhưng cho bộ **Communities and Crime**
(`communities_processed.csv`, target = `ViolentCrimesPerPop`, **100 feature số**).
So sánh R²: **baseline GBM (full)** vs **Boruta (RF)** vs **Lasso**.

Đây là kịch bản **"nhiều feature nhiễu"** (≈18/100 feature |corr|<0,1) → kỳ vọng selection **giúp ích rõ** cho model tuyến tính.

In [1]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from boruta import BorutaPy

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/AIO-Conquer02')
os.getcwd()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/AIO-Conquer02'

In [3]:
import pandas as pd
df = pd.read_csv('communities_processed.csv')
print('Shape:', df.shape)
df.info(verbose=False)

Shape: (1994, 101)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Columns: 101 entries, population to ViolentCrimesPerPop
dtypes: float64(101)
memory usage: 1.5 MB


### Chuẩn bị dữ liệu

Bộ này **đã processed thành toàn số** (không có cột chữ → **không cần encode**). Chỉ tách X/y và chia train/test.
Target `ViolentCrimesPerPop` giữ **nguyên gốc**.

In [4]:
target_colum = 'ViolentCrimesPerPop'
X = df.drop(columns=[target_colum], errors='ignore')
y = df[target_colum]

# Chia Train/Test trước để tránh data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Số feature:', X.shape[1])

Số feature: 100


## Baseline — Gradient Boosting với toàn bộ feature

In [5]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

gbr = GradientBoostingRegressor(max_depth=5, random_state=42)
gbr.fit(X_train, y_train)
preds = gbr.predict(X_test)
r2_score_all = round(r2_score(y_test, preds), 3)
print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")

Chỉ số R2-score khi sử dụng tất cả feature: 0.609


## Boruta (lõi Random Forest)

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from boruta import BorutaPy

# --- Bước 1: Tên feature ban đầu ---
feature_names = X.columns.tolist()

# --- Bước 2: Chuyển sang Numpy để tránh lỗi Index ---
X_train_np = X_train.values if hasattr(X_train, 'columns') else X_train
X_test_np = X_test.values if hasattr(X_test, 'columns') else X_test
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# --- Bước 3: Mô hình lõi ---
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

# --- Bước 4: Cấu hình & chạy Boruta ---
boruta_selector = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    verbose=0,
    random_state=42,
    max_iter=100
)
boruta_selector.fit(X_train_np, y_train_np)

# --- Bước 5: Lọc feature được chọn ---
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test_np)

# --- Bước 6: Huấn luyện lại trên feature đã lọc ---
rf_final = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
rf_final.fit(X_train_filtered, y_train_np)
preds_boruta = rf_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test_np, preds_boruta), 3)

confirmed_features = [feature_names[i] for i, c in enumerate(boruta_selector.support_) if c]

print("="*60)
print(f"Chỉ số R2-score khi sử dụng Boruta: {r2_score_boruta}")
print(f"Số lượng tính năng được giữ lại: {len(confirmed_features)} / {len(feature_names)}")
print(f"Các tính năng được chọn: {confirmed_features}")
print("="*60)

Chỉ số R2-score khi sử dụng Boruta: 0.599
Số lượng tính năng được giữ lại: 18 / 100
Các tính năng được chọn: ['racepctblack', 'racePctWhite', 'pctWInvInc', 'PctPopUnderPov', 'MalePctDivorce', 'FemalePctDiv', 'TotalPctDiv', 'PctFam2Par', 'PctKids2Par', 'NumIlleg', 'PctIlleg', 'PctSpeakEnglOnly', 'PctLargHouseFam', 'PctPersDenseHous', 'PctHousLess3BR', 'HousVacant', 'NumStreet', 'PctUsePubTrans']


## Lasso (LassoCV, có chuẩn hoá)

In [7]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np

# --- Bước 1: Chuẩn hoá (Lasso cần cùng thang đo để phạt công bằng) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bước 2: LassoCV tự tìm alpha tối ưu ---
lasso = LassoCV(
    alphas=np.logspace(-4, 1, 100),
    cv=5,
    max_iter=10000,
    tol=0.01,
    random_state=42,
    n_jobs=-1
)
lasso.fit(X_train_scaled, y_train)

# --- Bước 3: Dự đoán & R2 ---
preds_lasso = lasso.predict(X_test_scaled)
r2_score_lasso = round(r2_score(y_test, preds_lasso), 3)

# --- Bước 4: Feature bị Lasso loại (hệ số = 0) ---
feature_names = X.columns if hasattr(X, 'columns') else [f"Feature_{i}" for i in range(X.shape[1])]
coefficients = lasso.coef_
kept_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef != 0]
dropped_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef == 0]

# --- Bước 5: So sánh với baseline ---
print("="*60)
print(f"[-] R2-score Baseline (Gradient Boosting): {r2_score_all}")
print(f"[+] R2-score khi sử dụng Lasso: {r2_score_lasso}")
print(f" Alpha tối ưu: {round(lasso.alpha_, 5)}")
print("="*60)
print(f" Số feature bị Lasso loại (hệ số = 0): {len(dropped_features)} / {len(feature_names)}")
if len(dropped_features) > 0:
    print(f"   Bị loại: {dropped_features}")
print(f"\n Số feature giữ lại: {len(kept_features)}")
print("="*60)

[-] R2-score Baseline (Gradient Boosting): 0.609
[+] R2-score khi sử dụng Lasso: 0.642
 Alpha tối ưu: 0.00023
 Số feature bị Lasso loại (hệ số = 0): 17 / 100
   Bị loại: ['population', 'agePct12t21', 'agePct16t24', 'medIncome', 'perCapInc', 'TotalPctDiv', 'PersPerFam', 'PctImmigRec8', 'PctImmigRec10', 'PctRecImmig5', 'PctRecImmig10', 'PctHousOwnOcc', 'MedYrHousBuilt', 'OwnOccMedVal', 'RentMedian', 'PctSameHouse85', 'PopDens']

 Số feature giữ lại: 83


## Kết quả & nhận xét

Số liệu **GBR/Lasso đã kiểm chứng local** (deterministic vì cố định `random_state=42` → chạy Colab ra y hệt). Boruta chạy trên Colab.

| Phương pháp | R² (test) | Feature giữ lại |
|---|---|---|
| **Baseline — Gradient Boosting (full)** | 0,609 | 100/100 |
| **Lasso (LassoCV)** | **0,642** | **83/100** (bỏ 17, alpha≈0,0002) |
| **Boruta (RF)** | 0.599 | 18 / 100 |

**🎯 Điểm mấu chốt:** đây là bộ mà **feature selection thật sự có impact dương** lên model tuyến tính — **Lasso (0,642) vượt cả GBM (0,609)** trong khi **bỏ hẳn 17 feature**. Số 17 feature bị loại **trùng khớp** với ~18 feature `|corr|<0,1` (gần nhiễu) đã thấy ở notebook EDA → đúng tinh thần *"ít feature hơn mà tổng quát tốt hơn"*.

Khác hẳn 2 bộ housing/life trước (selection không cắt được gì): bộ này **nhiều cột nhiễu thật** → Lasso gỡ đúng phần nhiễu mà Linear Regression bị tổn thương. Feature đã chuẩn hoá [0,1] nhưng vẫn `StandardScaler` cho Lasso để phạt công bằng.